In [4]:
from glob import glob
import pandas as pd
glob("../data/interim/*_my_*transform*")

['../data/interim\\12_my_larva_transform.csv',
 '../data/interim\\17_my_creep_transform.csv',
 '../data/interim\\21_my_supply_transform.csv',
 '../data/interim\\25_my_economy_transform.csv',
 '../data/interim\\29_my_army_transform.csv',
 '../data/interim\\33_my_tech_transform.csv',
 '../data/interim\\4_my_larva_born_transform.csv',
 '../data/interim\\8_my_unspent_transform.csv']

In [5]:

glob("../data/interim/*_pro_*transform*")

['../data/interim\\12_pro_larva_transform.csv',
 '../data/interim\\17_pro_creep_transform.csv',
 '../data/interim\\21_pro_supply_transform.csv',
 '../data/interim\\25_pro_economy_transform.csv',
 '../data/interim\\29_pro_army_transform.csv',
 '../data/interim\\33_pro_tech_transform.csv',
 '../data/interim\\4_pro_larva_born_transform.csv',
 '../data/interim\\8_pro_unspent_transform.csv']

In [20]:
import matplotlib.pyplot as plt
from utils import *


pro_df = local.read.csv("4_pro_larva_born_transform", 'interim')
pro_terran_df = pro_df.query('opponent_race == "Terran"')
pro_terran_aggregated_data = pro_terran_df.groupby('second')['larva_born_count'].quantile(0.80)[:-50]
pro_terran_score = sum(pro_terran_aggregated_data)
# print(",".join(list([str(x) for x in pro_terran_aggregated_data.values])))

pro_zerg_df = pro_df.query('opponent_race == "Zerg"')
pro_zerg_aggregated_data = pro_zerg_df.groupby('second')['larva_born_count'].quantile(0.80)[:-50]
pro_zerg_score = sum(pro_zerg_aggregated_data)
# print(",".join(list([str(x) for x in pro_zerg_aggregated_data.values])))

pro_protoss_df = pro_df.query('opponent_race == "Protoss"')
pro_protoss_aggregated_data = pro_protoss_df.groupby('second')['larva_born_count'].quantile(0.80)[:-50]
pro_protoss_score = sum(pro_protoss_aggregated_data)
# print(",".join(list([str(x) for x in pro_protoss_aggregated_data.values])))







In [21]:
pro_protoss_score


39045.0

In [14]:
my_df = local.read.csv("4_my_larva_born_transform", 'interim')
my_df


,second,game_number,larva_born_count,opponent_race
0,0,1,0,Terran
1,1,1,0,Terran
2,2,1,0,Terran
3,3,1,0,Terran
4,4,1,0,Terran
...,...,...,...,...
140173,594,331,186,Zerg
140174,595,331,187,Zerg
140175,596,331,188,Zerg
140176,597,331,188,Zerg


In [17]:
my_df2 = local.read.csv("3_extract_my_larva_borns", "interim")
my_df2


,game_number,opponent_race,game_length_seconds_max_600,larva_born_time,main_player_won
0,1,Terran,600,10.446429,Win
1,1,Terran,600,20.803571,Win
2,1,Terran,600,30.714286,Win
3,1,Terran,600,40.625000,Win
4,1,Terran,600,50.535714,Win
...,...,...,...,...,...
44247,331,Zerg,600,590.401786,Win
44248,331,Zerg,600,594.553571,Win
44249,331,Zerg,600,595.178571,Win
44250,331,Zerg,600,598.258929,Win


In [44]:
player_wons = []
larva_borns_scores = []
game_numbers = []
matchups = []
for game_number in range(1, 333):
    temp_df = my_df[my_df['game_number'] == game_number].copy()
    temp_df2 = my_df2[my_df2['game_number'] == game_number].copy()
    if temp_df.shape[0] == 0:
        continue
        
    player_wons.append(1 if temp_df2.iloc[0].main_player_won == 'Win' else 0)
    matchup = temp_df2.iloc[0].opponent_race
    game_numbers.append(game_number)
    matchups.append(matchup)
    max_second = max(temp_df.second)
    if max_second < 420:
        larva_borns_scores.append(-1)
    else:
        my_score = float(temp_df.larva_born_count.sum())
        if matchup == "Terran":
            pro_score = sum(pro_terran_df[pro_terran_df['second'] <= max_second].groupby('second')['larva_born_count'].quantile(0.80))
        elif matchup == 'Zerg':
            pro_score = sum(pro_zerg_df[pro_zerg_df['second'] <= max_second].groupby('second')['larva_born_count'].quantile(0.80))
        else:
            pro_score = sum(pro_protoss_df[pro_protoss_df['second'] <= max_second].groupby('second')['larva_born_count'].quantile(0.80))
        if pro_score == 0:
            larva_borns_scores.append(1.0)
        else:
            larva_borns_scores.append(my_score/pro_score)

In [48]:
output_df = pd.DataFrame()
output_df['game_number'] = game_numbers
output_df['matchup'] = matchups
output_df['player_won'] = player_wons
output_df['larva_borns_score'] = larva_borns_scores
output_df


,game_number,matchup,player_won,larva_borns_score
0,1,Terran,1,0.862458
1,2,Zerg,1,0.960730
2,3,Terran,1,0.896749
3,4,Terran,0,0.918522
4,5,Terran,1,0.867607
...,...,...,...,...
238,326,Protoss,1,0.941193
239,327,Terran,1,0.935845
240,328,Zerg,1,0.840811
241,330,Zerg,1,0.823088
